In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import pandas as pd
from tqdm import tqdm

import _prompt
import _mapping
import _util
from _intervention import get_label_probability

In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A6000
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

In [4]:
model_type = "GPT-OSS_vanilla" # GPT-OSS or R1

if "GPT-OSS" in model_type:
    model, tokenizer = _util.load_OSS()
elif "R1" in model_type:
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
prompt_type = "h" # empty or pre_result or pre_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h_prompts{prompt_type[2:]}.csv")
else:
    prompts = pd.read_csv(f"data/{model_type}/prompts{prompt_type}.csv")
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} divided prompts")

loaded 256 divided prompts


In [10]:
intervention_loc = [11] # restatement or reasoning or restatement_and_reasoning

if type(intervention_loc) == str:
    if 'h' in prompt_type:
        if model_type == "GPT-OSS_stepwise":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit_h
        elif model_type == "GPT-OSS_vanilla":
            intervention_ids_dict = _mapping.intervene_ids_vanilla_2_digit_h
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit_h
        else:
            raise ValueError(f"Invalid model type: {model_type}")
    else:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit
        else:
            raise ValueError(f"Invalid model type: {model_type}")

    if intervention_loc == "restatement":
        intervention_ids = intervention_ids_dict["restatement"]
    elif intervention_loc == "reasoning":
        intervention_ids = intervention_ids_dict["reasoning"]
    elif intervention_loc == "restatement_and_reasoning":
        intervention_ids = intervention_ids_dict["restatement"] + intervention_ids_dict["reasoning"]
elif type(intervention_loc) == list:
    intervention_ids = intervention_loc

print(intervention_ids)

[11]


In [11]:
# Get header of divided prompts dataset
header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'intervend_prompt', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']

filepath = _util.create_csv_file(f"experiments/token_intervention/output/{model_type}/steps", f"{prompt_type[1:]}_{'_'.join(str(id) for id in intervention_ids)}.csv", header, overwrite=False)

batch_size = 24

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    factual_labels_str = [str(output) for output in batch_rows['base_sum'].tolist()]
    factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)
    counterfactual_labels_str = [str(output) for output in batch_rows['source_sum'].tolist()]
    counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)
    
    # Prepare batch of intervention prompts
    intervention_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    input_length = tokens.input_ids.shape[1]

    # Forward pass for the entire batch
    with torch.no_grad():
        output = model(input_ids=tokens.input_ids, attention_mask=tokens.attention_mask)

    pred_toks = output.logits[:,-1,:].argmax(dim=-1)
    prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
    factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
    counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
    tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
    del output
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        intervention_prompt = intervention_prompts[j]
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), intervention_prompt, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:27<00:00,  2.49s/it]


In [7]:
# Get header of divided prompts dataset
header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'intervend_prompt', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']

filepath = _util.create_csv_file(f"experiments/token_intervention/output/{model_type}/steps", f"{prompt_type[1:]}_{'_'.join(str(id) for id in intervention_ids)}.csv", header, overwrite=False)

batch_size = 24

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    factual_labels_str = [str(output) for output in batch_rows['factual_output'].tolist()]
    factual_labels = tokenizer(factual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)
    counterfactual_labels_str = [str(output) for output in batch_rows['counterfactual_output'].tolist()]
    counterfactual_labels = tokenizer(counterfactual_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)
    
    # Prepare batch of intervention prompts
    intervention_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    input_length = tokens.input_ids.shape[1]

    # Forward pass for the entire batch
    with torch.no_grad():
        output = model(input_ids=tokens.input_ids, attention_mask=tokens.attention_mask)

    pred_toks = output.logits[:,-1,:].argmax(dim=-1)
    prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
    factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
    counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
    tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
    del output
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        intervention_prompt = intervention_prompts[j]
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), intervention_prompt, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])


  0%|                                                                                                                        | 0/11 [00:00<?, ?it/s]


KeyError: 'factual_output'